In [13]:
from common_utils.utils_Sep import *
from pathlib import Path
import common_utils.CONFIG as C


In [14]:
st='2026-02-23 08:00:00'
et='2026-02-24 08:00:00' # 一定要是00:00:00 不然23:59:59:901的会被过滤掉
symbol = 'POWER'
last_time = pd.to_datetime('2026-02-24 08:00:00')
lookback_window = 1
lookback_days = 1

In [15]:
def load_funding_diffs(symbol: str, last_time, spread_bid: pd.Series, spread_ask: pd.Series,
                       lookback_days=4, mode='BN-OKX',OnSettleTime = True):
    """
    返回：
      funding_diff: DataFrame(index=FundingTime, cols=['FundingRate_okx','FundingRate_binance','funding_diff'])
      merged_df:    含 Type1 结果（funding_diff_adj_cumsum）
      merged_df2:   含 Type2 结果（funding_diff_signed_cumsum）
    依赖：
      process_funding_time_v3(csv_path, exchange)
      你的资金费率 CSV 路径结构
      
    Parameters:
    -----------
    symbol : str, 交易对符号
    last_time : datetime, 结束时间
    spread_bid : pd.Series, bid价差序列
    spread_ask : pd.Series, ask价差序列
    lookback_days : int, 回看天数，默认4天
    mode : str, 交易所模式，如 'BN-OKX', 'BN-BYBIT', 'BN-GATE' 等
    """
    symbol = symbol.upper()
    
    # 根据 mode 选择对应的交易所和文件路径
    if mode == 'BN-OKX':
        exchange2_csv = f"{C.FUNDING_RATE_OKX_DIR}/{symbol}-USDT-SWAP.csv"
        exchange1_csv = f"{C.FUNDING_RATE_BINANCE_DIR}/{symbol}USDT.csv"
        exchange2_name = 'okx'
        exchange1_name = 'binance'
    elif mode == 'BN-BYBIT':
        exchange2_csv = f"{C.FUNDING_RATE_BYBIT_DIR}/{symbol}USDT.csv"
        exchange1_csv = f"{C.FUNDING_RATE_BINANCE_DIR}/{symbol}USDT.csv"
        exchange2_name = 'bybit'
        exchange1_name = 'binance'
    elif mode == 'BN-GATE':
        exchange2_csv = f"{C.FUNDING_RATE_GATE_DIR}/{symbol}USDT.csv"
        exchange1_csv = f"{C.FUNDING_RATE_BINANCE_DIR}/{symbol}USDT.csv"
        exchange2_name = 'gate'
        exchange1_name = 'binance'
    else:
        # 默认使用 BN-OKX
        exchange2_csv = f"{C.FUNDING_RATE_OKX_DIR}/{symbol}-USDT-SWAP.csv"
        exchange1_csv = f"{C.FUNDING_RATE_BINANCE_DIR}/{symbol}USDT.csv"
        exchange2_name = 'okx'
        exchange1_name = 'binance'

    start_time = last_time - pd.Timedelta(days=lookback_days)

    df_okx     = process_funding_time_v3(exchange2_csv, exchange2_name)
    df_binance = process_funding_time_v3(exchange1_csv, exchange1_name)

    df_b = df_binance[(df_binance['Time'] >= start_time) & (df_binance['Time'] < last_time)].copy()
    df_o = df_okx[(df_okx['Time'] >= start_time) & (df_okx['Time'] < last_time)].copy()

    # 去重，仅保留每个 FundingTime 最后一条（并去掉最后一期未结算点）
    if OnSettleTime:
        df_o = df_o.drop_duplicates(subset='FundingTime', keep='last')[:-1]
        df_b = df_b.drop_duplicates(subset='FundingTime', keep='last')[:-1]


    # 对齐不同结算周期
    funding_interval_bn  = int((df_b.iloc[-1]['FundingTime'] - df_b.iloc[-2]['FundingTime']).total_seconds() / 3600)
    funding_interval_okx = int((df_o.iloc[-1]['FundingTime'] - df_o.iloc[-2]['FundingTime']).total_seconds() / 3600)

    df_o = df_o.rename(columns={'FundingRate': 'FundingRate_okx'})
    df_b = df_b.rename(columns={'FundingRate': 'FundingRate_binance'})
    
    if OnSettleTime:
        if funding_interval_bn == funding_interval_okx:
            funding_diff = (
                df_b[['FundingTime', 'FundingRate_binance']].set_index('FundingTime')
                .join(df_o[['FundingTime', 'FundingRate_okx']].set_index('FundingTime'), how='left')
            )
        elif funding_interval_bn > funding_interval_okx:
            df_o_agg = (
                df_o.set_index('FundingTime')
                .resample(f'{funding_interval_bn}h', label='right', closed='right')['FundingRate_okx']
                .sum().to_frame()
            )
            funding_diff = df_b[['FundingTime', 'FundingRate_binance']].set_index('FundingTime').join(df_o_agg, how='left')
        else:
            df_b_agg = (
                df_b.set_index('FundingTime')
                .resample(f'{funding_interval_okx}h', label='right', closed='right')['FundingRate_binance']
                .sum().to_frame()
            )
            funding_diff = df_o[['FundingTime', 'FundingRate_okx']].set_index('FundingTime').join(df_b_agg, how='left')
    else:
        funding_diff = pd.merge_asof(
            df_b[['Time', 'FundingRate_binance']].sort_values('Time'),
            df_o[['Time', 'FundingRate_okx']].sort_values('Time'),
            on='Time',
            direction='nearest',
            tolerance=pd.Timedelta(seconds=30)
        ).set_index('Time')

    funding_diff['funding_diff'] = funding_diff['FundingRate_okx'] - funding_diff['FundingRate_binance']


    # ---------- Type1：中位数规则 ----------
    spread_resampled = spread_bid.copy().to_frame()
    spread_resampled.columns = ['spread_bid_resampled']
    spread_resampled['spread_ask_resampled'] = spread_ask

    merged_df = pd.merge_asof(funding_diff, spread_resampled, left_index=True, right_index=True,tolerance=pd.Timedelta(seconds=90),direction='nearest')
    med_bid = spread_bid.quantile(0.5)
    med_ask = spread_ask.quantile(0.5) 

    merged_df['sign'] = np.where(
        merged_df['spread_ask_resampled'] > med_ask,  1,
        np.where(merged_df['spread_bid_resampled'] < med_bid, -1, 1)
    )
    merged_df['funding_diff_adj'] = merged_df['funding_diff'] * merged_df['sign']
    merged_df['funding_diff_adj_cumsum'] = merged_df['funding_diff_adj'].cumsum()

    # ---------- Type2：p40 / p60 规则 ----------
    merged_df2 = merged_df.copy()
    fd      = merged_df2['funding_diff']
    spr_bid = merged_df2['spread_bid_resampled']
    spr_ask = merged_df2['spread_ask_resampled']

    p40 = spread_bid.quantile(0.1)
    p60 = spread_ask.quantile(0.9)
    merged_df2['p40'] = p40
    merged_df2['p60'] = p60
    merged_df2['bid_50'] = med_bid
    merged_df2['ask_50'] = med_ask
    merged_df2['sign_rule'] = np.where(
        ((fd >= 0) & (spr_bid >= p40)) | ((fd <= 0) & (spr_ask >= p60)),
        1,
        np.where(fd == 0, 0, -1)
    ).astype('int8')

    merged_df2['funding_diff_signed'] = fd * merged_df2['sign_rule']
    merged_df2['funding_diff_signed_cumsum'] = merged_df2['funding_diff_signed'].cumsum()

    return funding_diff, merged_df, merged_df2


In [16]:
import plotly.graph_objects as go

# 1) 价差
cf_depth_resampled, spread_bid, spread_ask = load_spread_data_history_data(symbol, st, et, mode='BN-BYBIT')

# 2) 资金费率（含 Type1 / Type2）
funding_diff, merged_df, merged_df2 = load_funding_diffs(
    symbol, last_time, spread_bid, spread_ask, lookback_window, mode='BN-BYBIT', OnSettleTime=False
)
merged_df = merged_df.dropna()
merged_df2 = merged_df2.dropna()

# 3) Plotly 画图 —— Spread / Funding Diff / Bid Price (两个交易所)
bid_px_ex0 = cf_depth_resampled['bid_price_1_ex0']
bid_px_ex1 = cf_depth_resampled['bid_price_1_ex1']
start_time = last_time - pd.Timedelta(days=lookback_window) + pd.Timedelta(minutes=10)
funding_sorted = funding_diff.sort_index()

fig = go.Figure()

# Spread (y1, 左轴)
fig.add_trace(go.Scatter(
    x=spread_bid.index, y=spread_bid.values,
    name='Spread', line=dict(color='#66C2A5', width=1.6),
    yaxis='y1',
))

# Funding Diff (y2, 右轴1)
fig.add_trace(go.Scatter(
    x=funding_sorted.index, y=funding_sorted['funding_diff'].values,
    name='Funding Diff', line=dict(color='#FC8D62', width=1.8),
    yaxis='y2',
))

# Market1 Bid Price (y3, 右轴2)
bid_px_ex0_aligned = bid_px_ex0.reindex(spread_bid.index)
fig.add_trace(go.Scatter(
    x=spread_bid.index, y=bid_px_ex0_aligned.values,
    name='Market1 Bid (px)', line=dict(color='#8DA0CB', width=1.2),
    yaxis='y3',
))

# Market2 Bid Price (y3, 同轴)
bid_px_ex1_aligned = bid_px_ex1.reindex(spread_bid.index)
fig.add_trace(go.Scatter(
    x=spread_bid.index, y=bid_px_ex1_aligned.values,
    name='Market2 Bid (px)', line=dict(color='#E78AC3', width=1.2),
    yaxis='y3',
))

fig.update_layout(
    title=f"{symbol}  —  Spread / Funding / Bid   [{start_time.strftime('%Y-%m-%d %H:%M')}  →  {last_time.strftime('%Y-%m-%d %H:%M')}]",
    xaxis=dict(title='Time', domain=[0, 0.85]),
    yaxis=dict(title='Spread', side='left', showgrid=True),
    yaxis2=dict(title='Funding Diff', side='right', overlaying='y', showgrid=False),
    yaxis3=dict(title='Bid Price', side='right', overlaying='y', position=0.95, showgrid=False),
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.8)'),
    height=800, width=1600,
    template='plotly_white',
)
fig.show()

results = {
    "symbol": symbol,
    "funding_diff_signed_cumsum": merged_df2['funding_diff_signed_cumsum'].iloc[-1],
    "funding_diff_adj_cumsum": merged_df2['funding_diff_adj_cumsum'].iloc[-1],
}